# 10 · Compare Estimators

Summary table and visualization: where does each design land relative to the true β = 0.06?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel  = pd.read_parquet(DATA_DIR / "ship_month_panel.parquet")
events = pd.read_parquet(DATA_DIR / "category_change_events.parquet")

print(f"Panel: {panel.shape}  |  Events: {len(events)}")

In [ ]:
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

## Re-run all estimators (summary)

In [ ]:
# 1. Naive OLS
m_ols = smf.ols("log_revenue_per_berth ~ cabin_category_count", data=panel).fit(cov_type="HC3")
beta_ols = m_ols.params["cabin_category_count"]

# 2. OLS + controls
m_ctrl = smf.ols(
    "log_revenue_per_berth ~ cabin_category_count + np.log(berth_capacity)"
    " + C(brand_tier) + C(sailing_region)", data=panel
).fit(cov_type="HC3")
beta_ctrl = m_ctrl.params["cabin_category_count"]

# 3. TWFE
panel_idx = panel.set_index(["ship_id", "month"])
panel_idx["log_berth"] = np.log(panel_idx["berth_capacity"])
fe2 = PanelOLS(
    panel_idx["log_revenue_per_berth"],
    panel_idx[["cabin_category_count", "log_berth"]],
    entity_effects=True, time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)
beta_fe = fe2.params["cabin_category_count"]

print("Estimators assembled. Building comparison table...")

In [ ]:
# 4. Event study average post-effect — load from saved or recompute inline
# Quick inline version (post-period mean of dynamic coefficients)
panel_es = panel.merge(events[["ship_id", "event_month_idx"]], on="ship_id", how="left")
panel_es["month_idx"] = panel_es.groupby("ship_id")["month"].transform(
    lambda s: (pd.to_datetime(s).dt.to_period("M") - pd.to_datetime(s).min().to_period("M")).apply(lambda x: x.n)
)
panel_es["rel_t"] = panel_es["month_idx"] - panel_es["event_month_idx"]
panel_es["treated"] = panel_es["event_month_idx"].notna()

WINDOW = 12
ALL_PERIODS = [k for k in range(-WINDOW, WINDOW + 1) if k != -1]
panel_es_idx = panel_es.set_index(["ship_id", "month"])
for k in ALL_PERIODS:
    col = f"dum_{k}"
    panel_es_idx[col] = ((panel_es_idx["treated"]) &
                         (panel_es_idx["rel_t"].clip(-WINDOW, WINDOW) == k)).astype(float)

dum_cols = [f"dum_{k}" for k in ALL_PERIODS]
es_model = PanelOLS(
    panel_es_idx["log_revenue_per_berth"],
    panel_es_idx[dum_cols],
    entity_effects=True, time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

post_dum_cols = [f"dum_{k}" for k in range(1, WINDOW + 1)]
beta_es = es_model.params[post_dum_cols].mean()

print(f"Event study post-period average: {beta_es:.4f}")

In [ ]:
# Full comparison table
import pandas as pd

avg_delta = events["delta_categories"].mean()
true_beta = 0.06
true_att  = true_beta * avg_delta  # ATT scales by average Δ categories

rows = [
    ("Naive OLS",           beta_ols,  "Associational — confounded by scale"),
    ("OLS + controls",      beta_ctrl, "Partial adjustment — scale residual remains"),
    ("Two-way FE",          beta_fe,   "Within-ship variation; removes scale FE"),
    ("Event Study (avg)",   beta_es,   "Dynamic DiD; uses refit timing"),
    ("True β (per cat.)",   true_beta, "Known DGP parameter"),
    ("True ATT (β × Δ̄)",   true_att,  f"Expected avg effect given Δ̄ = {avg_delta:.2f}"),
]

df = pd.DataFrame(rows, columns=["Estimator", "Estimate", "Description"])
df["Bias vs β"] = df["Estimate"].apply(lambda x: f"{(x - true_beta)/true_beta*100:+.0f}%")
print(df.to_string(index=False))

## Visualization

In [ ]:
labels = [r[0] for r in rows]
coefs  = [r[1] for r in rows]
colors = ["#E74C3C", "#E67E22", "#2ECC71", "#3498DB", "#95A5A6", "#95A5A6"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, coefs, color=colors, alpha=0.8, edgecolor="white")
ax.axvline(true_beta, color="black", ls="--", lw=1.5, label=f"True β = {true_beta}")
ax.set_xlabel("Estimated Coefficient on Cabin Category Count")
ax.set_title("Estimator Comparison\nCross-sectional OLS → Panel FE → Event Study vs Ground Truth")
ax.legend()

for bar, coef in zip(bars, coefs):
    ax.text(coef + 0.002, bar.get_y() + bar.get_height()/2,
            f"{coef:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "10_estimator_comparison.png", bbox_inches="tight")
plt.show()

## Key takeaways

1. **Naive OLS** overstates the effect by 5–10× because of scale confounding.
2. **Controls** help but don't fix the problem — unobserved scale is the core issue.
3. **Panel FE** removes time-invariant scale, getting close to the true β.
4. **Event study** uses refit timing quasi-experimentally; post-period average aligns with FE.
5. **The gap** between OLS and FE estimates = scale selection effect (the bias we're accounting for).

In interview terms: FE and event study are your identification strategy. OLS is the baseline you're improving upon. The progression from OLS to FE to event study is the story.